In [ ]:
import pandas as pd
ratings=pd.read_csv("/Users/harshverma/Downloads/machine learning /movie recommendation system/ml-latest-small/ratings.csv")
movies =pd.read_csv("/Users/harshverma/Downloads/machine learning /movie recommendation system/ml-latest-small/movies.csv")

In [12]:
# finding the average rating of the movies 
avg_rating = ratings.groupby('movieId')['rating'].mean()
rating_count = ratings.groupby('movieId')['rating'].count()
movie_stats = pd.DataFrame({
    'avg_rating': avg_rating,
    'rating_count': rating_count
}).reset_index()
movie_stats = movie_stats.merge(movies[['movieId', 'title']],on='movieId')
movie_stats.sort_values('avg_rating', ascending=False).head(10)


,movieId,avg_rating,rating_count,title
7638,88448,5.0,1,Paper Birds (Pájaros de papel) (2010)
8089,100556,5.0,1,"Act of Killing, The (2012)"
9065,143031,5.0,1,Jump In! (2007)
9076,143511,5.0,1,Human (2015)
9078,143559,5.0,1,L.A. Slasher (2015)
4245,6201,5.0,1,Lady Jane (1986)
8136,102217,5.0,1,Bill Hicks: Revelations (1993)
8130,102084,5.0,1,Justice League: Doom (2012)
4240,6192,5.0,1,Open Hearts (Elsker dig for evigt) (2002)
9104,145994,5.0,1,Formula of Love (1984)


In [20]:
#weighted rating formula 
C = ratings['rating'].mean()
m = movie_stats['rating_count'].quantile(0.91)

print("C:", C)
print("m:", m)
#imdb formula 
# weighted rating applying the formula
# wr= v/(v+m)R + m/(v+m)C
movie_stats['weighted_rating'] = (
    (movie_stats['rating_count'] / (movie_stats['rating_count'] + m)) * movie_stats['avg_rating']
    + (m / (movie_stats['rating_count'] + m)) * C)
qualified = movie_stats[movie_stats['rating_count'] >= m]
top_10 = qualified.sort_values('weighted_rating', ascending=False).head(10)
top_10[['title', 'avg_rating', 'rating_count', 'weighted_rating']]

C: 3.501556983616962
m: 30.0


,title,avg_rating,rating_count,weighted_rating
277,"Shawshank Redemption, The (1994)",4.429022,317,4.348838
659,"Godfather, The (1972)",4.289062,192,4.182643
2224,Fight Club (1999),4.272936,218,4.179624
224,Star Wars: Episode IV - A New Hope (1977),4.231076,251,4.153191
46,"Usual Suspects, The (1995)",4.237745,204,4.143362
461,Schindler's List (1993),4.225000,220,4.138187
257,Pulp Fiction (1994),4.197068,307,4.135153
897,Star Wars: Episode V - The Empire Strikes Back...,4.215640,211,4.126750
1938,"Matrix, The (1999)",4.192446,278,4.125152
921,"Godfather: Part II, The (1974)",4.259690,129,4.116646


In [26]:
# to avoid temporal leak we use timestamp in sorted order  
ratings_sorted = ratings.sort_values('timestamp').reset_index(drop=True)
split_index = int(len(ratings_sorted) * 0.8)
train = ratings_sorted.iloc[:split_index]
test = ratings_sorted.iloc[split_index:]
print("shape of test",train.shape,"shape of train",test.shape)
# range of timestamp in which train and test works 
print("Train date range:", pd.to_datetime(train['timestamp'], unit='s').min(), "to", pd.to_datetime(train['timestamp'], unit='s').max())
print("Test date range:", pd.to_datetime(test['timestamp'], unit='s').min(), "to", pd.to_datetime(test['timestamp'], unit='s').max())

shape of test (80668, 4) shape of train (20168, 4)
Train date range: 1996-03-29 18:36:55 to 2016-03-22 08:26:02
Test date range: 2016-03-22 08:26:11 to 2018-09-24 14:27:30


In [28]:
#weighted rating formula for train set only  
train_avg_rating = train.groupby('movieId')['rating'].mean()
train_rating_count = train.groupby('movieId')['rating'].count()

train_movie_stats = pd.DataFrame({
    'avg_rating': train_avg_rating,
    'rating_count': train_rating_count
}).reset_index()
C_train = train['rating'].mean()
m_train = train_movie_stats['rating_count'].quantile(0.91)
#imdb formula 
# weighted rating applying the formula
# wr= v/(v+m)R + m/(v+m)C
train_movie_stats['weighted_rating'] = (
    (train_movie_stats['rating_count'] / (train_movie_stats['rating_count'] + m_train)) * train_movie_stats['avg_rating']
    + (m_train / (train_movie_stats['rating_count'] + m_train)) * C_train
)
print("C_train:", C_train)
print("m_train:", m_train)
train_movie_stats.head()

C_train: 3.5084791986909307
m_train: 30.0


,movieId,avg_rating,rating_count,weighted_rating
0,1,3.923913,184,3.865675
1,2,3.400000,95,3.426035
2,3,3.294118,51,3.373511
3,4,2.357143,7,3.290659
4,5,3.063830,47,3.237070


In [30]:
# creating a dictionary 
movie_weighted_rating_lookup = train_movie_stats.set_index('movieId')['weighted_rating'].to_dict()
def predict_baseline(movie_id):
    return movie_weighted_rating_lookup.get(movie_id, C_train)
test = test.copy()
test['predicted_rating'] = test['movieId'].apply(predict_baseline)
unseen_count = (~test['movieId'].isin(movie_weighted_rating_lookup.keys())).sum()
print(f"Test rows with unseen movies (fallback to C): {unseen_count} / {len(test)} ({unseen_count/len(test):.2%})")

Test rows with unseen movies (fallback to C): 3044 / 20168 (15.09%)


In [31]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse = np.sqrt(mean_squared_error(test['rating'], test['predicted_rating']))
mae = mean_absolute_error(test['rating'], test['predicted_rating'])

print(f"Baseline RMSE: {rmse:.4f}")
print(f"Baseline MAE: {mae:.4f}")

Baseline RMSE: 1.0380
Baseline MAE: 0.8130


In [ ]:
#Baseline RMSE: 1.038 it should come lower this is the maximum worst performance 
# Baseline MAE:  0.813. offset we can except from the predictions 
# Coverage gap:  15.09% (fallback to global mean) percentage of movies that is not seen by model
#item based cosine 
# RMSE: 1.0769
# MAE:  0.8495
# User-Based CF (Pearson):
# RMSE: 1.0758
# MAE:  0.8511
# Item-Based KNNWithMeans:
# RMSE: 1.0689
# MAE:  0.8438
# k=10: RMSE=1.0145, MAE=0.7821
# k=20: RMSE=1.0141, MAE=0.7820
# k=50: RMSE=1.0145, MAE=0.7827
# k=100: RMSE=1.0137, MAE=0.7832
# k=150: RMSE=1.0151, MAE=0.7840